### Foundations of Finance
# Python Practice #5: Portfolios

Write your full name and ID below:

STUDENT 1: Willem Elshout 

STUDENT 2: (group member)

---
## 1. **Run:** Prepare the environment

Let's start by importing (and installing when needed) all the necessary libraries for the task:
- [math](https://docs.python.org/3/library/math.html): built-in math stuff.
- [numpy](https://numpy.org/): numerical methods.
- [pandas](https://pandas.pydata.org/): data manipulation.
- [plotly](https://plotly.com/): interactive plots.
- [random](https://docs.python.org/3/library/random.html): random number generator, sampling.
- [scipy](https://scipy.org/): scientific package.
- [statsmodels](https://www.statsmodels.org): statistical models.

All required packages must be installed before they are imported. The installation procedure depends on your specific Python setup (e.g., Anaconda). Using the default package manager `pip` should work in most cases.  

In [ ]:
from math import sqrt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import random
from scipy.optimize import minimize
from scipy.stats import uniform
import statsmodels.api as sm

---
## 2. **Run:** define your portfolio

It is a good practice to have all the **configuration parameters** in a single place, near the top of the notebook, for easy access.

This cell configures everything in the notebook. Don't worry: the actual meaning of every parameter will become clear as you advance through the practice.

You may specify here, among other things:
- The **composition of the portfolio**.
- The **time period** under consideration.
- The **risk-free rate**.

In addition to that, there are a few **technical parameters** that you may change if you understand what they do *(It is safe to leave those unchanged)*.

In [ ]:
# Risk-free asset (can be changed)
RISK_FREE_RATE = 0.25 / 100

# Assets in YOUR portfolio (can be changed)
PF_ASSETS = ['GOLD', 'GRF', 'REP', 'ITX']

# count items in portfolio
PF_NUM_ASSETS = len(PF_ASSETS)

# Index name (do NOT change, there is only one index in the dataset)
INDEX_NAME = 'IBEX35TR'
PF_ASSETS_AND_INDEX = PF_ASSETS + [INDEX_NAME]

# Dates (can be changed)
PF_SINCE = '20130101'
PF_UNTIL = '20191231'

# Plot configuration (be careful)
RANDOM_PORTFOLIOS = 10000
CML_STEPS = 100
CML_EXCESS = 75
POWER_TRICK = 3

---
# 3. Loading data

In this exercise, we have three data sources:
- **IBEX35_constituents.xlsx:** List of IBEX 35 index constituents.
- **IBEX35_prices.xlsx:** Historical prices (in euros) of index constituents and two "versions" of the index (i.e., *IBEX 35*, and *IBEX 35 Total Return*).
- **GOLD_prices.xlsx:** Historical price of LBMA gold (in euros). 

**GOOGLE COLAB:** If you are using Google Colab, use the "folder" icon on the leftmost side of the window to open the filesystem. Then, drag and drop there the Excel files.

## 3.1. **Run:** load index constituents

Read an Excel file. In this case, the list of constituents of the Spanish index IBEX 35.

In [ ]:
index_constituents = pd.read_excel('IBEX35_constituents.xlsx', index_col='symbol')
index_constituents

## 3.2. **Exercise:** Read stock price history

For your convenience, the file `IBEX35_prices.xlsx` has been downloaded from Factset and contains enough historical prices for this exercise.  
Write the code that reads stock prices into a *DataFrame* named `index_prices`. Remember to use column `date` as index.  

(about 1 line of code)

In [ ]:
# Reading IBEX35 stock prices into a DataFrame
index_prices = pd.read_excel('IBEX35_prices.xlsx', index_col='date')


## 3.3. **Exercise:** Read gold price history

For your convenience, the file `GOLD_prices.xlsx` has been downloaded from Factset and contains enough historical prices for this exercise.  
Write the code that reads gold prices into a *DataFrame* named `gold_prices`. Remember to use column `date` as index.  

(about 1 line of code)

In [ ]:
# Reading gold prices into a DataFrame
gold_prices = pd.read_excel('GOLD_prices.xlsx', index_col='date')


## 3.4. **Exercise:** Join both tables into a single dataset

Finally, let's join everything (i.e., IBEX35 prices, and gold prices) into a new `original_prices` *DataFrame*.  
Ensure the left side of the join is `prices_index` and the right `gold_prices`.  
The result should use the index on the **left side** of the join (which is the default behaviour).

**Hint:** use `join()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.join.html))

(about 1 line of code)

In [ ]:
# Joining the two DataFrames
original_prices = index_prices.join(gold_prices)

#print(original_prices)


## 3.5. **Run:** Select a time period

The dataset is quite large, let's focus on a specific time period between the dates `PF_SINCE` and `PF_UNTIL`.  
Selecting a portion of a dataset is commonly referred as **subsetting**.

In [ ]:
# date range
selected_dates = (original_prices.index > PF_SINCE) & (original_prices.index < PF_UNTIL)

# stocks and the index
all_prices = original_prices.loc[ selected_dates, PF_ASSETS_AND_INDEX ]

# just the stocks
prices = original_prices.loc[ selected_dates, PF_ASSETS ]

## 3.6. **Run:** Are there any missing values?

Sometimes there are missing values of gaps in our time series. If that is the case, you may apply different strategies such as dropping or imputing them.  
Let's check with `isna()`: `False` is good, as it means that there are not missing values in the columns.  

(about 1 line of code)

In [ ]:
all_prices.isna().any(axis=0)

---
# 4. Exploring the data

## 4.1. **Exercise:** General description of the tables

Use the function `describe()` to get some statistics on the data stored in `all_prices` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.describe.html))

(1 line of code)

In [ ]:
all_prices.describe()


## 4.2. **Run:** Plot historical prices

Let's create interactive plots with [plotly](https://plotly.com).  
For example, let's plot the price of any item by its name.

In [ ]:
def plot_asset_price(prices, name):
    # plot prices using plotly timeseries plot
    fig = px.line(prices, x=prices.index, y=name, title=f'price of {name}')
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()

Let's plot the **IBEX 35 Total Return**.

In [ ]:
plot_asset_price(all_prices, INDEX_NAME)

And **gold** as well.

In [ ]:
plot_asset_price(all_prices, 'GOLD')

---
# 5. Calculate returns

## 5.1. **Exercise:** Calculate discrete returns

The formula for **discrete returns**:
$$R_t = \frac{P_t - P_{t-1}+ D_t}{P_{t-1}}= \frac{P_t - P_{t-1}}{P_{t-1}} + \frac{D_t}{P_{t-1}}$$  

**However**, since we have **adjusted prices**, the dividend term does not apply.  

Calculate the returns of `prices` and store them in a new variable `daily_discrete_returns`.

**Hint:** use `pct_change()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.pct_change.html)).

(about 1 line of code)

In [ ]:
daily_discrete_returns = all_prices.pct_change()


## 5.2. **Exercise:** Calculate continuous returns

The formula for **continuous returns**:
$$R_t = \log \left( \frac{P_t}{P_{t-1}} \right)$$  

**This is what we will use in our calculations!** Calculate the log-returns of `prices` and store them in a new variable `daily_continuous_returns`.

**Hint:** use `np.log()` for the natural logarithm, and `shift()` for the lagged prices ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.shift.html)).

(about 1 line of code)

In [ ]:
daily_continuous_returns = np.log(all_prices / all_prices.shift(1))


## 5.3. **Run:** Plot histograms of prices and returns

Let's plot the histograms of prices and returns.

**QUESTION:** Which one is expected to look like a normal distribution (more or less)?

Pick the first asset in the portfolio (or any other you like)

In [ ]:
ASSET = prices.columns[0]
ASSET

Plot the histogram of **Prices**.

In [ ]:
fig = px.histogram(prices, x=ASSET)
fig.show()

Plot the histogram of **Returns**.

In [ ]:
fig = px.histogram(daily_continuous_returns, x=ASSET)
fig.update_layout(xaxis= { 'tickformat': '.1%'})
fig.show()

---
# 6. Annual **Expected Return** and **Volatility**

Markowitz model relies on expected returns and volatilities (as a measure of risk). Let's calculate them:

## 6.1. **Exercise:** Annual Expected Return

First, we calculate the daily expected return as the average of daily returns:

$$E(R_i)= \mu = \frac{1}{T} \sum_{t=1}^T R_{it}$$

By convention, a year has 252 trading days, so the **annual expected return** is: 

$$E(R_{year}) = 252 \cdot E(R_{day})$$

Calculate annual expected returns using `mean()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html)) on `daily_continuous_returns`.

Store the result in a new variable `expected_rets`.

(about 1 line of code)

In [ ]:
expected_rets = daily_continuous_returns.mean() * 252


## 6.2. **Exercise:** **Standard deviation** (also known as **volatility**) is a measure of risk:

$$\sigma^2 = \frac{1}{T} \sum_{t=1}^T {(R_t - \mu)}^2$$

$$\sigma = \sqrt{\sigma^2}$$

**NOTE:** Uses $T$ rather than $T-1$ in the formula because it is not a sample.

Regarding the **annual volatility**: as mentioned earlier, we have 252 trading days in a year. That's an international agreement, a convention for returns and risks:

$$\sigma^2_{year} = 252 \cdot {\sigma^2_{day}}$$

$$\sigma_{year} = \sqrt{252} \cdot {\sigma_{day}}$$


Calculate annual volatilities using `std()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.std.html)) on `daily_continuous_returns`. Don't forget the factor!  

Store the result in a new variable `vols`.


(about 1 line of code)

In [ ]:
vols = daily_continuous_returns.std() * np.sqrt(252)


## 6.3 **Exercise:** calculate the covariance matrix

The covariance matrix is equal to:

$$\left[ \sigma_{ij} \right] = \rho_{ij} \cdot \sigma_i \cdot \sigma_j$$

Again, multiply by 252 to get annual figures.

Calculate the covariance matrix using `cov()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.cov.html)) on `daily_continuous_returns`.  

Store the result in a new variable `cov_matrix`.

(about 1 line of code)

In [ ]:
cov_matrix = daily_continuous_returns.cov() * 252


## 6.4. **Run:** Plot matrices

Let's plot the matrices. To make plots clearer, display numerical values with just 3 decimal places.

In [ ]:
cov_matrix_plot = cov_matrix.round(3)

Heatmap of the **covariance matrix**.

In [ ]:
fig = px.imshow(cov_matrix_plot, text_auto=True, width=600, height=600)
fig.show()

Heatmap of **correlation matrix**.  

Calculate it using `corr()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html))

In [ ]:
corr_matrix = daily_continuous_returns.corr()
corr_matrix_plot = corr_matrix.round(3)
fig = px.imshow(corr_matrix_plot, text_auto=True, zmin=-1, zmax=+1, width=600, height=600)
fig.show()

---
# 7. Basic portfolio calculations

Some theorical concepts to refresh before you move forward.

- Portfolio's **expected return**:

$$E(r_p) = \sum_{i=1}^n{w_i E(r_i)}$$

- Portfolio's **volatility**:

$$\sigma_p^2 = \sum_{i=1}^n \sum_{j=1}^n {w_i w_j \sigma_i \sigma_j \rho_{ij}}$$


- And remember Tobin's formula:

$$E(R_p) = r_f + \frac{E(R_i) - r_f}{\sigma_i} \cdot \sigma_p$$

- **Sharpe ratio**: the risk premium is the rate of return on an investment over and above the risk-free or guaranteed rate of return. To calculate risk premium, investors must first calculate the estimated return and the risk-free rate of return.

$$\texttt{sharpe}_p = \frac{R_p - r_f}{\sigma_p}$$

## 7.1. **Run:** some useful functions to make your life easier

Some of the formulae above can be translated into very simple Python code! Let's express them first as **vectors** and **matrices**:


$$E(r_p) = \vec w_i \cdot \vec R_i$$

$$\sigma_p^2 = \vec w_i \cdot \left[ \sigma_{ij} \right] \cdot {\vec w_i}^T$$

Check the result below, in functions `portfolio_return()` and `portfolio_volatility()`. Isn't it cool?


In [ ]:
def daily_returns(prices):
    #return prices.pct_change()
    return np.log(prices / prices.shift(1))

def expected_returns(prices):
    return 252 * daily_returns(prices).mean()

def covariance_matrix(prices):
    return 252 * daily_returns(prices).cov()

def correlation_matrix(prices):
    return daily_returns(prices).corr()

def volatilities(prices):
    return sqrt(252) * daily_returns(prices).std()

def portfolio_return(weights, expected_returns):
    return weights @ expected_returns

def portfolio_volatility(weights, cov_matrix):
    return sqrt(weights @ cov_matrix @ weights.T)

def portfolio_sharpe_ratio(portfolio_return, portfolio_volatility, risk_free_rate):
    return (portfolio_return - risk_free_rate) / portfolio_volatility

## 7.2. **Exercise:** Let's test them with some trivial weights

Feel free to customize the `weights`, however, remember the following constraints:

$$0 \le w_i \le 1$$

$$\sum_{i=1}^n w_i = 1$$

This is the procedure to calculate the $E(r_p)$, $\sigma_p$ and $\texttt{sharpe}_p$ of a portfolio given historical prices and weights $w_i$:
1. Calculate expected returns.
2. Calculate volatilities.
3. Calculate covariance matrix.
4. Call `portfolio_return()`
5. Call `portfolio_volatility()`
6. Call `portfolio_sharpe_ratio()`

Here are steps 1, 2, and 3 (given to you)

In [ ]:
# create an example weight vector (this can be changed)
weights = pd.Series([1./PF_NUM_ASSETS] * PF_NUM_ASSETS, index=PF_ASSETS)
weights

In [ ]:
returns = expected_returns(prices)
returns

In [ ]:
vols = volatilities(prices)
vols

In [ ]:
cov_matrix  = covariance_matrix(prices)
cov_matrix

**Now write the code for steps 4, 5, and 6.**  
Each step requires just a function call.  
**Hint:** you will need `RISK_FREE_RATE`, which has been defined on the top of the notebook.  

Store the results in `pf_ret`, `pf_vol`, and `pf_sharpe`.  

(about 3 lines of code, 1 per call)

In [ ]:
# Step 4: Calculate Portfolio Return
pf_ret = portfolio_return(weights, returns)

# Step 5: Calculate Portfolio Volatility
pf_vol = portfolio_volatility(weights, cov_matrix)

# Step 6: Calculate Portfolio Sharpe Ratio
pf_sharpe = portfolio_sharpe_ratio(pf_ret, pf_vol, RISK_FREE_RATE)


And finally, print the results: (given to you)

In [ ]:
# display results
print('For this allocation:')
print(weights)
print(f'\nResulting portfolio:\n return={pf_ret:.2%}\n volatility={pf_vol:.2%}\n sharpe={pf_sharpe:.2}')

---
# 8. The "Brute force" approach

How would you calculate the **tangent portfolio** and the **minimum-risk portfolio** if you had not any advanced tool?  

A possible answer would be:
1. Generate *a lot* of random portfolios, and for each of them calculate *return*, *volatility* and *Sharpe ratio*.
2. Look for the portfolio with the **maximum Sharpe ratio** $\rightarrow$ the **tangent portfolio**.
3. Look for the portfolio with the **minimum volatility** $\rightarrow$ the **minimum-risk portfolio**.  

This is a "brute force" approach, however, given a high enough number of tries, you will find pretty accurate results!

## 8.1. **Run:** Create random portfolios

This is a bit trickier: all possible weights combinations (and hence portfolios) conform a shape in the $R_p - \sigma_p$ plane. Let's "explore" that surface by generating a lot (i.e. thousands) of random portfolios. The result would be a cloud of points that we will plot later.

Again, remember the constraints:

$$0 \le w_i \le 1$$

$$\sum_{i=1}^n w_i = 1$$

The cell below will create a lot of portfolios (the exact amount can be configured in `RANDOM_PORTFOLIOS`) with **randomly generated weights**.

In [ ]:
def create_random_portfolios(number, asset_names):
    
    # create table of random weights
    random_table = uniform.rvs(size=(number, len(asset_names)))
    random_table = random_table ** POWER_TRICK
    portfolios = pd.DataFrame(random_table, columns=asset_names)

    # enforce each row sums 1.0
    for i in portfolios.index:
        wi = portfolios.loc[i]
        sum_weights = wi.sum()
        portfolios.loc[i] = wi / sum_weights 
        
    return portfolios

portfolios = create_random_portfolios(RANDOM_PORTFOLIOS, PF_ASSETS)
portfolios.tail()

## 8.2. **Run:** Calculate portfolio metrics for each set of random weights
For each random combination of weights $w_i$, calculate the metrics of the corresponding portfolio: $R_p$ , $\sigma_p$ , and $\texttt{sharpe}$.

In [ ]:
def calculate_metrics(portfolios, asset_names):
    for i in portfolios.index:
        w_i = portfolios.loc[i, asset_names]
        portfolios.loc[i, 'return'    ] = pf_ret = portfolio_return(w_i, returns)
        portfolios.loc[i, 'volatility'] = pf_vol = portfolio_volatility(w_i, cov_matrix)
        portfolios.loc[i, 'sharpe'    ] = portfolio_sharpe_ratio(pf_ret, pf_vol, RISK_FREE_RATE)
    
    return portfolios
    
portfolios = calculate_metrics(portfolios, PF_ASSETS)
portfolios.tail()

## 8.3. **Run:** Quick plot
This is a quick plot so you may see how we are advancing.  
We will improve it **a lot** in the next cells...

- **X axis:** volatility.
- **Y axis:** expected return.
- **Color:** the sharpe ratio.

**QUESTION:** With this visual aid, can you spot the **minimum risk** and **maximum sharpe** portfolios? Can you identify the original assets?

In [ ]:
# Plotly scatter plot 
px.scatter(
    portfolios,
    x='volatility',
    y='return',
    color='sharpe',
    width=600,
    height=600)

## 8.4. **Run:** Numerical approximation to portfolio with **maximum Sharpe ratio**

This is a "quick & dirty" approximation to the portfolio with the maximum Sharpe ratio, also known as the **tangent portfolio**.

Let's look for the row with the **maximum Sharpe ratio** in the table of **randomly generated portfolios**. Again, this is not exact!

This function:
1. Finds the row with the maximum value in column `sharpe`.
2. For that specific row, prints the asset weights and the columns `return`, `volatility`, `sharpe`.

**Hint:** Check how the function `idxmax()` works ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.idxmax.html))

In [ ]:
def find_max_sharpe(portfolios):

    # find index of the row with the top value
    idx_max_sharpe = portfolios['sharpe'].idxmax()
    ms_pf = portfolios.loc[idx_max_sharpe]

    # retrieve values
    max_sharpe         = ms_pf['sharpe']
    max_sharpe_ret     = ms_pf['return']
    max_sharpe_vol     = ms_pf['volatility']
    max_sharpe_weights = ms_pf.drop(['sharpe', 'return', 'volatility'])

    # print results
    print('RESULT OBTAINED BY RANDOMLY GENERATED PORTFOLIOS')
    print(f'Max(sharpe): portfolio at index {idx_max_sharpe}, return={max_sharpe_ret:.2%} volatility={max_sharpe_vol:.2%} sharpe={max_sharpe:.2f}')
    print(max_sharpe_weights)
    
find_max_sharpe(portfolios)

## 8.5. **Exercise:** Numerical approximation to portfolio with **minimum risk**

Same idea, but looking for **minimum risk**. Again, this is approximate!

Write a fuction that:
1. Finds the row with the minimum value in column `volatility`.
2. For that specific row, prints the asset weights and the columns `return`, `volatility`, `sharpe`.

**Hint:** Check the function `idxmin()` ([documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.idxmin.html))

In [ ]:
def find_min_risk(portfolios):
    # Find index of the row with the minimum volatility
    idx_min_risk = portfolios['volatility'].idxmin()
    min_risk_pf = portfolios.loc[idx_min_risk]

    # Retrieve values
    min_risk         = min_risk_pf['volatility']
    min_risk_ret     = min_risk_pf['return']
    min_risk_sharpe  = min_risk_pf['sharpe']
    min_risk_weights = min_risk_pf.drop(['sharpe', 'return', 'volatility'])

    # Print results
    print('RESULT OBTAINED BY RANDOMLY GENERATED PORTFOLIOS')
    print(f'Min(risk): portfolio at index {idx_min_risk}, return={min_risk_ret:.2%} volatility={min_risk:.2%} sharpe={min_risk_sharpe:.2f}')
    print(min_risk_weights)

find_min_risk(portfolios)

---
# 9. The right way: usage of optimization routines

## 9.1. **Run:** Use optimization to calculate tangent portfolio

Python can optimize any function in a similar way to the exercise done in class in Excel.

Let's use function `minimize(...)` in package **scipy**.  

**Coding Trick:** As we need to *maximize* Sharpe ratio ($\texttt{sharpe}$) and the aforementioned function only *minimizes*, the way forward is to *minimize* the negative of Sharpe ratio ($-\texttt{sharpe}$), right?

So `scipy.optimize.minimize(...)` takes as arguments:
- The function to minimize, $-\texttt{sharpe}$, written as Python code in `_neg_sharpe_ratio(...)`.
- The `initial_guess`, in our case we are allocating the capital evenly along the portolio.
- Arguments to pass to `_neg_sharpe_ratio(...)`
- Optimization method.
- Variable boundaries: $0 \le w_i \le 1$
- Constraints: $\sum w_i - 1 = 0$

**THIS CODE IS COMPLEX:** you must understand what it does, but don't bother with the technicalities.

In [ ]:
def _neg_sharpe_ratio(weights, expected_returns, cov_matrix, risk_free_rate):
    # negative sharpe ratio
    
    pf_ret    = portfolio_return(weights, returns)
    pf_vol    = portfolio_volatility(weights, cov_matrix)
    pf_sharpe = portfolio_sharpe_ratio(pf_ret, pf_vol, risk_free_rate)
    
    return - pf_sharpe


def maximize_sharpe_ratio(expected_returns, cov_matrix, risk_free_rate):
    # This function maximizes sharpe ratio for given data
    
    num_assets = len(expected_returns)
    initial_guess = num_assets*[1./num_assets,]
    args = (expected_returns, cov_matrix, risk_free_rate)
    
    # all weights must add up to 1.0
    constraints = ({'type': 'eq', 'fun': lambda x: x.sum() - 1})
    
    # bounded by 0 and 1
    bound = (0.0,1.0) 
    bounds = tuple(bound for asset in range(num_assets))
    
    # optimize
    result = minimize(_neg_sharpe_ratio,
                      initial_guess,
                      args=args,
                      method='SLSQP',
                      bounds=bounds,
                      constraints=constraints)
    
    print(result.message)
    
    return pd.Series(result.x, index=expected_returns.index).sort_values(ascending=False)

Let's use the function to find the solution to our specific problem.  
**IMPORTANT:** In the next section, an exercise will require you to do something similar to this, so pay attention...)

In [ ]:
# optimize
max_sharpe_weights = maximize_sharpe_ratio(returns, cov_matrix, RISK_FREE_RATE)

# calculate return and volatility
max_sharpe_ret = portfolio_return(max_sharpe_weights, returns)
max_sharpe_vol = portfolio_volatility(max_sharpe_weights, cov_matrix)

# print results
print('OPTIMIZATION')
print(f'The tangent portfolio has return={max_sharpe_ret:.1%} and volatility={max_sharpe_vol:.1%}')
print('And the following composition:')
for i in max_sharpe_weights.index:
    print(f'{i}\t{max_sharpe_weights[i]:03.1%}')

## 9.2. **Exercise**: Use optimization to calculate minimum risk portfolio

Same procedure, but in this case we minimize risk.

**THIS CODE IS COMPLEX:** you must understand what it does, but don't bother with the technicalities.

In [ ]:
# This function minimizes risk for given data
def minimize_risk(cov_matrix):
    num_assets = len(cov_matrix)
    initial_guess = num_assets*[1./num_assets,]
    args = (cov_matrix,)
    # all weights must add up to 1.0
    constraints = ({'type': 'eq', 'fun': lambda x: x.sum() - 1})
    # bounded by 0 and 1
    bound = (0.0,1.0) 
    bounds = tuple(bound for asset in range(num_assets))
    result = minimize(portfolio_volatility,
                      initial_guess,
                      args=args,
                      method='SLSQP',
                      bounds=bounds,
                      constraints=constraints)
    
    print(result.message)
    
    return pd.Series(result.x, index=cov_matrix.columns).sort_values(ascending=False)

Now, getting inspiration from the equivalent cell from above, **write the code that solves for the minimum risk portfolio.**  
1. Call `minimize_risk()` instead of `maximize_sharpe_ratio()`.
2. With the weights of the minimum risk portfolio, calculate return and volatility.
3. Display the results in the same manner.


(about 8 lines of code)

In [ ]:
# Call minimize_risk to find the weights of the minimum risk portfolio
min_risk_weights = minimize_risk(cov_matrix)

# Calculate return and volatility of the minimum risk portfolio
min_risk_ret = portfolio_return(min_risk_weights, returns)
min_risk_vol = portfolio_volatility(min_risk_weights, cov_matrix)

# Print results
print('OPTIMIZATION')
print(f'The minimum risk portfolio has return={min_risk_ret:.1%} and volatility={min_risk_vol:.1%}')
print('And the following composition:')
for i in min_risk_weights.index:
    print(f'{i}\t{min_risk_weights[i]:03.1%}')



---
# 10. Efficient frontier: the interactive plot

## 10.1 **Run:** Interactive plot

The following code builds an interactive plot which displays:
- The original assets.
- A cloud of points with all the random portfolios we have just generated. The color is the $\texttt{sharpe}$ ratio.
- The Capital Market Line, based on the data points available.
- Tangent portfolio.
- Minimum-risk portfolio.

The plot is interactive:
- For any given point, it displays the $w_i$ along $E\left(r_p\right)$ and $\sigma_p$ of that asset combination.
- In the CML, it also displays the percentage invested in the risk-free asset.
- You may zoom in/out, save image to disk, etc.

**THE CODE IN THIS SECTION IS COMPLEX:** visualization techniques are **out of the scope** of the practice.

In [ ]:
def calculate_cml(max_sharpe_ret, max_sharpe_vol, risk_free_rate, cml_steps, cml_excess):
    
    cml = pd.DataFrame()

    cml['weight'] = range(cml_steps + cml_excess) 
    cml['weight'] /= cml_steps
    cml['volatility'] = cml['weight'] * max_sharpe_vol
    cml['return'] = risk_free_rate + (max_sharpe_ret - risk_free_rate) * cml['volatility']/max_sharpe_vol
    # 'hover' is the nice text that appears when the cursor hovers over the CML points
    cml['hover'] = [f'<br>{w_i:.1%}' for w_i in cml['weight']]
    
    return cml

In [ ]:
# calculate CML
cml = calculate_cml(max_sharpe_ret, max_sharpe_vol, RISK_FREE_RATE, CML_STEPS, CML_EXCESS)

In [ ]:
def write_allocation_text(portfolios, asset_names):
    # write the text that appears when the cursor hovers over the portfolios
    portfolios['hover'] = ''
    for i in portfolios.index:
        wi = portfolios.loc[i, prices.columns]
        portfolios.loc[i, 'hover'] = '<br>'.join([f'{company}:{wi[company]:.0%}' for company in asset_names])
        
    return portfolios

In [ ]:
# prepare texts
portfolios = write_allocation_text(portfolios, PF_ASSETS)

In [ ]:
def interactive_plot(portfolios, returns, vols, cal, max_sharpe_ret, max_sharpe_vol, min_risk_ret, min_risk_vol):
    # create base figure
    fig = go.Figure()

    # add random portfolios
    fig.add_trace(go.Scatter(
        x = portfolios['volatility'],
        y = portfolios['return'],
        mode='markers',
        text=portfolios['hover'],
        hovertemplate="<b>vol:%{x:.1%}<br>ret:%{y:.1%}<br></b>%{text}<extra></extra>",
        showlegend=True,
        marker=dict(
            color=portfolios['sharpe'],
            colorscale=['rgb(192, 192, 255)', 'rgb(255, 192, 192)'], #colorscale='Viridis',
            line_width=0
        ),
        name = 'portfolios'
    ))

    # add assets    
    fig.add_trace(go.Scatter(
        x=vols,
        y=returns,
        mode="markers+text",
        #name="Markers and Text",
        hovertemplate="<b>vol:%{x:.0%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text=vols.index,
        textposition="middle right",
        showlegend=True,
        marker=dict(color='black', size=10),
        name='assets'
    ))

    # add CML
    fig.add_trace(go.Scatter(
        x = cml['volatility'],
        y = cml['return'],
        mode='lines',
        text=cal['hover'],
        hovertemplate="<b>vol:%{x:.1%}<br>ret:%{y:.1%}<br></b>%{text}<extra></extra>",
        showlegend=True,
        marker=dict(
            color='red',
            line_width=0
        ),
        name = 'CML'
    ))

    # add tangent portfolio    
    fig.add_trace(go.Scatter(
        x=[max_sharpe_vol],
        y=[max_sharpe_ret],
        mode="markers+text",
        hovertemplate="<b>vol:%{x:.1%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text=['tangent portfolio'],
        textposition="middle left",
        showlegend=True,
        marker=dict(color='blue', size=10),
        name = 'Tangent'
    ))

    # add minimum risk portfolio    
    fig.add_trace(go.Scatter(
        x=[min_risk_vol],
        y=[min_risk_ret],
        mode="markers+text",
        hovertemplate="<b>vol:%{x:.1%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text=['minimum risk'],
        textposition="middle left",
        showlegend=True,
        marker=dict(color='blue', size=10),
        name='Minimum Vol.'
    ))
    
    # use percentages in axes
    fig.update_layout(
        width=800,
        height=800,
        xaxis= { 'tickformat': '.1%'},
        yaxis= { 'tickformat': '.1%'},
        title='Markowitz & Capital Market Line (CML)',
        xaxis_title='$\sigma_p$',
        yaxis_title='$R_p$',
    )

    fig.show()

Use the function defined above.

In [ ]:
# display the interactive plot
interactive_plot(portfolios,
                 returns, vols,
                 cml,
                 max_sharpe_ret, max_sharpe_vol,
                 min_risk_ret, min_risk_vol)

---
# 11. Beta estimation: CAPM and SML

The CAPM model:

$$E(R_i) = r_f +  \beta_i \cdot \left( E(R_M) - r_f \right)$$

Let's estimate a linear regression of every asset vs. the market.

## 11.1. **Run:** Fuction that plots de OLS
This function is given to you, as visualization techniques are **out of the scope** of the practice.

In [ ]:
def plot_ols_regression(x, y, title):
    fig = px.scatter(x=x, y=y, trendline='ols')
    fig.update_traces(line_color='red')
    fig.update_layout(
        width=400,
        height=400,
        xaxis= { 'tickformat': '.1%'},
        yaxis= { 'tickformat': '.1%'},
        title=title,
        xaxis_title='$R_M - R_f$',
        yaxis_title='$R_i - R_f$',
    )
    fig.show()

## 11.2. **Run:** CAPM calculation

The regression formula, solved with **Ordinary Least Squares** (OLS):

$$R_i(t) - r_f = \hat \alpha_i + \hat \beta_i \cdot \left( R_M(t) - r_f \right) + \epsilon_i$$

First, we have to calculate `daily_returns` and `expected_returns` for every asset and also the market itself.  
Next, the solve Ordinary Least Squares (OLS) regressions and assemble the results in a DataFrame.

**THE CODE IN THIS SECTION IS COMPLEX:** review it and try to understand what it does.

In [ ]:
def calculate_capm(prices, asset_list, index_name, risk_free_rate):

    # retrieve data
    capm_daily_rets  = daily_returns(prices)
    capm_exp_rets    = expected_returns(prices)
    capm_daily_rets  = capm_daily_rets.dropna()
    index_exp_return = capm_exp_rets[index_name]
    
    risk_free_rate_daily = risk_free_rate / 252

    # make room for the results
    results = pd.DataFrame(index=asset_list)
    
    for i, asset_name in enumerate(asset_list):
        # fore each asset in the list
        
        # Ordinary Least Squares regression (OLS)
        y = capm_daily_rets[asset_name] - risk_free_rate_daily
        X = capm_daily_rets[index_name] - risk_free_rate_daily
        X_cte = sm.add_constant(X)
        model = sm.OLS(y, X_cte)
        regression = model.fit()

        alpha= regression.params.iloc[0]
        beta = regression.params.iloc[1]
        r2 = regression.rsquared

        #if r2 < 0.5:
        #    # mean reversion
        #    adj_beta = 0.67 * beta + 0.33 * 1.00
        #else:
        #    # leave as it is
        #    adj_beta = beta

        # return estimated by CAPM
        capm_return = risk_free_rate + beta * (index_exp_return - risk_free_rate)

        # store results
        results.loc[asset_name, 'alpha'] = alpha
        results.loc[asset_name, 'beta']  = beta
        results.loc[asset_name, 'beta_pvalue'] = regression.pvalues.iloc[1]
        #results.loc[asset_name, 'adj. beta']  = adj_beta
        results.loc[asset_name, 'R2']  = r2
        results.loc[asset_name, 'expected_ret'] = capm_exp_rets[asset_name]
        results.loc[asset_name, 'capm_ret'] = capm_return
        
        # one plot per asset
        plot_ols_regression(X,
                            y,
                            f'OLS regression {asset_name} vs. {index_name}')
    
    return results, index_exp_return

Call the function defined above:

In [ ]:
capm_results, index_exp_return = calculate_capm(all_prices, PF_ASSETS, INDEX_NAME, RISK_FREE_RATE)
capm_results

## 11.3. **Exercise:** get annual results

The table `capm_results` above shows **daily figures**. Can you turn them into **annual figures**? (about 4 lines of code)

1. Copy `capm_results` into a new Dataframe, `annual_capm_results`.
2. Remember that the average number of trading days is 252, however, not every column needs to be multiplied by that factor!


**Hint:** to create a copy of a table, use `copy()`.

(about 3 lines of code)

In [ ]:
# Copy capm_results into a new DataFrame
annual_capm_results = capm_results.copy()

# Annualize relevant columns
annual_capm_results['alpha'] = annual_capm_results['alpha'] * 252
annual_capm_results['expected_ret'] = annual_capm_results['expected_ret'] * 252
annual_capm_results['capm_ret'] = annual_capm_results['capm_ret'] * 252

# The rest of the columns remain unchanged


---
# 12. **Run:** Plot the Security Market Line (SML)

This plot displays:
- The Security Market Line.
- The risk-free asset and the market.
- The assets in the portfolio.
- Their projection on the SML.

**THIS CODE IS COMPLEX:** visualization techniques are **out of the scope** of the practice.

In [ ]:
def plot_sml(sml, index_return, risk_free_rate):
    # create base figure
    fig = go.Figure()

    # add assets    
    fig.add_trace(go.Scatter(
        x=sml['beta'],
        y=sml['expected_ret'],
        mode="markers+text",
        #name="Markers and Text",
        hovertemplate="<b>beta:%{x:.0%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text=sml.index,
        textposition="middle right",
        showlegend=True,
        marker=dict(color='orange', size=10),
        name='assets'
    ))

    # add Security Market Line (and beyond)
    slope = index_return - risk_free_rate
    left = min(0, sml['beta'].min())
    right = max(1, sml['beta'].max())
    fig.add_trace(go.Scatter(
        x=[left, right],
        y=[risk_free_rate + left*slope, risk_free_rate + right*slope],
        mode="lines",
        showlegend=True,
        line=dict(dash='dot', color='gray'),
        name = 'SML'
    ))


    # add projections on Security Market Line    
    fig.add_trace(go.Scatter(
        x=sml['beta'],
        y=sml['capm_ret'],
        mode="markers+text",
        #name="Markers and Text",
        hovertemplate="<b>beta:%{x:.0%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text=sml.index,
        textposition="bottom right",
        showlegend=True,
        marker=dict(color='black', size=10),
        name = 'projections'
    ))

    # add market   
    fig.add_trace(go.Scatter(
        x=[1.0],
        y=[index_return],
        mode="markers+text",
        #name="Markers and Text",
        hovertemplate="<b>beta:%{x:.0%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text='market',
        textposition="top left",
        showlegend=True,
        marker=dict(color='red', size=10),
        name = 'market'
    ))

    # add risk-free asset
    fig.add_trace(go.Scatter(
        x=[0.0],
        y=[risk_free_rate],
        mode="markers+text",
        #name="Markers and Text",
        hovertemplate="<b>beta:%{x:.0%}<br>ret:%{y:.1%}<br></b><extra></extra>",
        text='risk-free',
        textposition="top left",
        showlegend=True,
        marker=dict(color='green', size=10),
        name = 'risk-free'
    ))

    # use percentages in axes
    fig.update_layout(
        width=800,
        height=800,
        #xaxis= { 'tickformat': '.1'},
        yaxis= { 'tickformat': '.0%'},
        title='Security Market Line (SML)',
        xaxis_title='$\\beta_i$',
        yaxis_title='$E(R_i)$',
    )

    fig.show()

Use the function defined above.  
Remember some key items in the SML, notation is $\left( \beta_i, r_i \right)$:
- $\left( 0, R_f \right)$ is the risk-free asset.
- $\left( 1, R_M \right)$ is the market portfolio.


In [ ]:
plot_sml(capm_results, index_exp_return, RISK_FREE_RATE)

---
# LIVE CODING SESSION (To be done in class)
---

# 13. **Run:** Setting everything up

Each team will solve a unique portfolio. The composition will be randomly picked for you.

**Write below your students IDs and sum them up.** That will be the seed of the random generator.

In [ ]:
SEED = 12345

Run the code below to pick three random assets. In addition to them, all teams will include **Gold** in their portfolios.

In [ ]:
POSSIBLE_ASSETS = [
    'ACS',  'ACX', 'AMS',  'ANA', 'BBVA', 'BKT',
    'CABK', 'COL', 'ELE',  'ENG', 'FDR', 'FER',
    'GRF',  'IAG', 'IBE',  'IDR', 'ITX', 'MAP',
    'MEL',  'MTS', 'NTGY', 'RED', 'REP', 'ROVI',
    'SAB',  'SAN', 'SCYR', 'SLR', 'TEF']

random.seed(SEED)
NEW_PF_ASSETS = ['GOLD'] + random.sample(POSSIBLE_ASSETS, 3)

print('CONSIDER THESE ASSETS IN THE LIVE CODE SESSION:')
print(f'{NEW_PF_ASSETS=}')

---
# 14. Answer the questions in the presentation/practice (this part is graded)

Use the spaces below if you need code to answer any of the questions.

**IMPORTANT:** needless to say, you are **allowed** (and **encouraged**) to call any of the functions previously defined in the notebook!

**IMPORTANT:** Do **NOT** copy/paste code when a **function call** would suffice!

In [ ]:
#
# WRITE YOUR CODE HERE
#

---
# The End